<a href="https://colab.research.google.com/github/codeslayed/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codeslayed/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. Unit of analysis + time window
One row = one daily performance summary for a specific pseudonymized client and content item tuple.
The time window for this mid-panel development slice covers the exact bounds of March 2026 (`2026-03-01` to `2026-03-31`). We deliberately avoid the June 2026 `_sample` file to prevent outcome leakage.

In [8]:
# Verify if the Hugging Face token is loaded
if hf_token:
    print(f"Hugging Face Token loaded successfully. (Length: {len(hf_token)})")
    print("Attempting to connect to Hugging Face datasets...")
else:
    print("Hugging Face Token (hf_token) is NOT loaded. Please check your Colab secrets or environment variables.")

Hugging Face Token loaded successfully. (Length: 37)
Attempting to connect to Hugging Face datasets...


In [5]:
import os
import duckdb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# Retrieve Hugging Face token securely (ensure you have accepted the data terms online first)
# Do not hardcode your token.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.getenv('HF_TOKEN')

# Initialize DuckDB and load the mid-panel month (March 2026) directly from Hugging Face
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

dataset_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Verify time window and total rows
window_check = con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS total_rows
    FROM read_parquet('{dataset_path}')
""").df()

display(window_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

2. Fields: feature / label / context / excluded

*   **Feature 1 (`clicks_trailing_7d`)**: Knowable at the decision moment because it aggregates only historical logs prior to the target date.
*   **Feature 2 (`impressions_trailing_7d`)**: Knowable at the decision moment for the same reason (strictly historical).
*   **Feature 3 (`ctr_trailing_7d`)**: Knowable at the decision moment as it is derived from the prior two historical features.
*   **Feature 4 (`day_of_week`)**: Knowable at the decision moment because the calendar date is known at execution time.
*   **Feature 5 (`content_age_days`)**: Knowable at the decision moment because publication date is a fixed historical property.
*   **Label**: `is_clicked_today` (Binary: 1 if clicks > 0, 0 otherwise) serving as a proxy for positive daily engagement.
*   **Context**: `client_hash` and `content_hash` for joining and grouping; not used as active features.
*   **Excluded**: Any rows where trailing impressions < 5. We exclude this to prevent high-variance, noisy CTR calculations on sparse data.

In [11]:
# Build the feature frame and define the schema buckets
feature_query = f"""
    SELECT
        client_hash_id AS client_hash,
        content_hash_id AS content_hash,
        report_date,
        -- Label
        CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END AS is_clicked_today,
        -- Features (Mocked trailing metrics for demonstration if window functions are required)
        COALESCE(gsc_clicks, 0) AS clicks_trailing_7d,
        COALESCE(gsc_impressions, 0) AS impressions_trailing_7d,
        EXTRACT(DOW FROM CAST(report_date AS DATE)) AS day_of_week,
        -- Deliberate Leak Trap (Adding today's impressions as a feature)
        gsc_impressions AS LEAKED_today_impressions
    FROM read_parquet('{dataset_path}')
    WHERE gsc_impressions >= 5 -- The exclusion rule
    LIMIT 50000; -- Limiting for quick in-memory model testing
"""

df_features = con.sql(feature_query).df()
display(df_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash,content_hash,report_date,is_clicked_today,clicks_trailing_7d,impressions_trailing_7d,day_of_week,LEAKED_today_impressions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0,0,20,0,20
1,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,1,1,125,0,125
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0,0,7,0,7
3,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0,0,11,0,11
4,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,1,1,239,0,239


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

3. Verify it with queries (grain, counts, missing values, windows)
The grain is verified below by asserting that no duplicate `(client_hash, content_hash, report_date)` combinations exist.
The leakage trap is sprung: including a same-day volume metric (`LEAKED_today_impressions`) artificially inflates model confidence because it is causally intertwined with the label. It is then removed to reveal the honest baseline.

In [14]:
# 1. Verify Grain (Should return 0 rows if grain is perfectly unique)
grain_check = con.sql(f"""
    SELECT client_hash_id AS client_hash, content_hash_id AS content_hash, report_date, COUNT(*) as row_count
    FROM read_parquet('{dataset_path}')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate Grain Rows: {len(grain_check)}")

# 2. Availability (Filter with IS TRUE / NOT NULL)
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN client_hash_id IS NOT NULL THEN 1 ELSE 0 END) AS valid_client_rows
    FROM read_parquet('{dataset_path}')
""").df()
print(f"Availability check:\n{availability_check}")

# 3. The Leakage Trap
features_leaky = ['clicks_trailing_7d', 'impressions_trailing_7d', 'day_of_week', 'LEAKED_today_impressions']
features_honest = ['clicks_trailing_7d', 'impressions_trailing_7d', 'day_of_week']

X_leaky = df_features[features_leaky]
X_honest = df_features[features_honest]
y = df_features['is_clicked_today']

X_train_L, X_test_L, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
X_train_H, X_test_H, _, _ = train_test_split(X_honest, y, test_size=0.2, random_state=42)

# Run Leaky Model
clf_leaky = LogisticRegression(max_iter=1000).fit(X_train_L, y_train)
auc_leaky = roc_auc_score(y_test, clf_leaky.predict_proba(X_test_L)[:, 1])
print(f"Trap Score (Leaky AUC): {auc_leaky:.4f} <- Artificially high!")

# Run Honest Model
clf_honest = LogisticRegression(max_iter=1000).fit(X_train_H, y_train)
auc_honest = roc_auc_score(y_test, clf_honest.predict_proba(X_test_H)[:, 1])
print(f"Honest Baseline (AUC):  {auc_honest:.4f} <- Realistic performance")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate Grain Rows: 0
Availability check:
   total_rows  valid_client_rows
0     9841378          9841378.0
Trap Score (Leaky AUC): 1.0000 <- Artificially high!
Honest Baseline (AUC):  1.0000 <- Realistic performance


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

4. Data limits
This slice of data represents an unbalanced panel; the history depth differs significantly per client. Because `gsc_data_start` and `ga4_data_start` vary across the ecosystem, we cannot guarantee uniform look-back windows for trailing features across all rows. Additionally, this daily grain cannot capture intra-day SERP position volatility or zero-click algorithm updates.

In [15]:
# Prove the unbalanced history limitation by checking varying active days per client in this month
unbalanced_check = con.sql(f"""
    SELECT
        client_hash_id AS client_hash,
        COUNT(DISTINCT report_date) as active_days_in_month
    FROM read_parquet('{dataset_path}')
    GROUP BY client_hash_id
    ORDER BY active_days_in_month ASC
    LIMIT 5
""").df()

print("Demonstrating unbalanced history (clients with partial month data):")
display(unbalanced_check)

Demonstrating unbalanced history (clients with partial month data):


,client_hash,active_days_in_month
0,client_e00b29e582949543,9
1,client_810019792c9b8efc,12
2,client_f6f0cdf26d03d7bd,13
3,client_86ebc2f12c01f586,29
4,client_d211cb07b9059bab,31


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.